# §7 Spine Ablation — Colab Runbook (v3, Drive-first, resumable)

Designed for unattended execution. Three properties make it survive disconnects:

1. **Drive-first.** All results write to `MyDrive/agent-runtime-results/` from the very first call. Nothing lives in ephemeral `/content/`.
2. **Resumable.** Per-scenario JSONL writes are fsync'd to disk. If the runtime dies at scenario 70 of 100, re-running picks up at scenario 71 with no double-spend.
3. **Single command per stage.** `run_full_ablation.py` orchestrates every cell of the schedule; you just pick `--stage smoke|lean|full`.

**Real talk about Colab and unattended runs.** Free Colab and Colab Pro both kill the runtime when the browser tab closes or after ~90 minutes of idle. Only Colab Pro+ has true background execution (~24h). If you don't have Pro+, the more reliable path for the lean and full stages is to run on your own Mac with `caffeinate + nohup`. See `evals/RUN_LOCALLY.md` in the repo. Use this notebook for smoke and as a Drive-backed checkpoint for shorter runs.

## Cell 1 — Mount Drive (do this first, before anything else)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RESULTS_DIR = '/content/drive/MyDrive/agent-runtime-results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Drive mounted. Results will write to:', RESULTS_DIR)
print('Existing files:', sorted(os.listdir(RESULTS_DIR)) or '(none)')

## Cell 2 — Clone repo and verify all new flags exist

In [ ]:
import os, subprocess
REPO = '/content/agent-runtime-patterns'
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/vasundras/agent-runtime-patterns.git', REPO], check=True)
help_text = subprocess.check_output(['python3', 'evals/run_eval.py', '--help'], cwd=REPO).decode()
for flag in ['--spine', '--model', '--live', '--mock', '--results-dir', '--heartbeat-every']:
    assert flag in help_text, f'Missing {flag} — push the latest evals/ to GitHub before continuing.'
assert os.path.isfile(os.path.join(REPO, 'evals/run_full_ablation.py')), 'run_full_ablation.py missing — push the latest commits.'
print('Verified. All flags and orchestrator present.')

In [ ]:
!pip install -q 'anthropic>=0.39' --upgrade

In [ ]:
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
assert os.environ['ANTHROPIC_API_KEY'].startswith('sk-ant-'), 'API key not loaded from Colab secrets'
print('API key loaded.')

## Cell 3 — Mock pre-flight (free)

Confirms the spine code path runs end-to-end before any paid call.

In [ ]:
MOCK_DIR = RESULTS_DIR + '/_mock'
!cd /content/agent-runtime-patterns && python3 evals/run_full_ablation.py --results-dir {MOCK_DIR} --stage smoke --mock
!ls -la {MOCK_DIR}
!wc -l {MOCK_DIR}/cost_ledger.jsonl

## Cell 4 — Smoke (≈$5, ~5 min)

P5 only, 5 scenarios, k=1, Sonnet 4.6. Writes directly to Drive. If your tab closes during this, just re-run the cell — already-done scenarios skip.

In [ ]:
!cd /content/agent-runtime-patterns && python3 evals/run_full_ablation.py --results-dir {RESULTS_DIR} --stage smoke --live

In [ ]:
# Verify the smoke results landed on Drive.
!ls -la {RESULTS_DIR}
!head -2 {RESULTS_DIR}/cost_ledger.jsonl

## Cell 5 — Lean (≈$72-120, ~5 hours)

30 scenarios, k=4, both spines, both models. 4 cells × 120 conversations = 480 total.

**At this duration on free Colab you will probably get disconnected.** That is fine — the run is resumable. To resume after a disconnect: re-run this cell. It will skip every scenario already in the JSONL files on Drive.

If you want to leave it overnight reliably, copy the command into your Mac terminal instead (see `RUN_LOCALLY.md` for the `caffeinate + nohup` recipe). Drive-mounted runs from your Mac are even more reliable than Colab Pro+.

In [ ]:
!cd /content/agent-runtime-patterns && python3 evals/run_full_ablation.py --results-dir {RESULTS_DIR} --stage lean --live

## Cell 6 — Decision gate

Show the lean Δpass^k before paying for the full run.

In [ ]:
!cd /content/agent-runtime-patterns && python3 evals/analyze.py --results-dir {RESULTS_DIR}

## Cell 7 — Full (≈$240-400, ~8-12 hours)

**Only run if Cell 6 supports the hypothesis.**

100 scenarios, k=4, both spines, both models. 4 cells × 400 conversations = 1,600 total. Resumable; safe to interrupt and re-launch.

In [ ]:
!cd /content/agent-runtime-patterns && python3 evals/run_full_ablation.py --results-dir {RESULTS_DIR} --stage full --live --auto-proceed

## Cell 8 — Final analysis

Run this any time to see the current state of results. Safe to run mid-experiment.

In [ ]:
!cd /content/agent-runtime-patterns && python3 evals/analyze.py --results-dir {RESULTS_DIR}

In [ ]:
from IPython.display import Markdown
Markdown(open(RESULTS_DIR + '/summary.md').read())

## Cell 9 (optional) — Live progress monitor

Tail the latest JSONL to see scenarios complete in real time. Safe to run while another cell is executing — Drive writes are visible immediately.

In [ ]:
import glob, os
files = sorted(glob.glob(RESULTS_DIR + '/*.jsonl'), key=os.path.getmtime)
if files:
    latest = files[-1]
    n = sum(1 for _ in open(latest))
    print(f'Latest: {os.path.basename(latest)} ({n} lines)')
    print(f'Tail:')
    !tail -3 {latest}
else:
    print('No JSONL files yet.')